# Observabilidad con Langfuse o Phoenix

Este notebook muestra el salto de una observabilidad manual a una plataforma especializada para agentes LLM.

## Objetivos
- Entender qué agrega una plataforma de observabilidad para agentes.
- Comparar logs caseros con trazas estructuradas.
- Instrumentar un agente con una capa de eventos similar a Langfuse/Phoenix.
- Revisar ejecuciones correctas y con error.

## ¿Qué aportan plataformas como Langfuse o Phoenix?

- Trazas visuales con jerarquía de spans.
- Sesiones y datasets de ejecuciones.
- Evaluaciones automáticas y comparaciones entre versiones.
- Contexto de prompts, respuestas, tokens, modelos y resultados.
- Alertas y debugging para errores y regressiones.

In [ ]:
!pip install pandas langchain langchain-openai wikipedia

In [ ]:
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_base=os.environ.get("GITHUB_BASE_URL"),
        openai_api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

In [ ]:
from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langchain_classic import hub
from langsmith import Client

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

client = Client()
prompt = client.pull_prompt("hormold/openai-functions-agent",
                            dangerously_pull_public_prompt=True)

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")

In [ ]:
import json
import time
from pprint import pprint

try:
    import pandas as pd
except ImportError:
    raise ImportError('Instala pandas con `pip install pandas` antes de ejecutar este notebook.')

# Simulador de cliente de observabilidad inspirado en plataformas como Langfuse o Phoenix
class ObservabilityPlatformClient:
    def __init__(self, project_name='demo-observability'):
        self.project_name = project_name
        self.events = []

    def log_event(self, event_type, payload):
        event = {
            'timestamp': time.time(),
            'event_type': event_type,
            'payload': payload,
            'event_id': str(time.time()).replace('.', '-') + '-' + event_type,
        }
        self.events.append(event)
        return event

    def get_events(self):
        return pd.DataFrame([
            {**e['payload'], 'event_type': e['event_type'], 'timestamp': e['timestamp']} for e in self.events
        ])

    def display(self):
        df = self.get_events()
        display(df)

# Agente instrumentado con plataforma de observabilidad
class PlatformAgent:
    def __init__(self, observability_client):
        self.obs = observability_client

    def execute(self, question, model_name='mock-llm-v2'):
        session_id = str(int(time.time() * 1000))
        self.obs.log_event('session.start', {'session_id': session_id, 'question': question})

        # Paso 1: prompt inicial
        prompt = question
        self.obs.log_event('prompt.recorded', {'session_id': session_id, 'prompt': prompt, 'model': model_name})

        # Paso 2: llamada al modelo
        start = time.time()
        response_text = f'Respuesta simulada para: {question[:60]}...'
        latency = time.time() - start
        tokens_in = len(prompt.split()) * 1.2
        tokens_out = min(80, len(response_text.split()))
        cost = round((tokens_in + tokens_out) * 0.00002, 6)

        self.obs.log_event('model.call', {
            'session_id': session_id,
            'model': model_name,
            'response': response_text,
            'latency_s': latency,
            'tokens_in': tokens_in,
            'tokens_out': tokens_out,
            'cost_estimated': cost,
        })

        # Paso 3: resultado y evaluación
        self.obs.log_event('response.generated', {
            'session_id': session_id,
            'final_answer': response_text,
            'status': 'ok',
        })

        return response_text

In [ ]:
# Crear cliente de observabilidad y agente
client = ObservabilityPlatformClient('il3.1-demo')
agent = PlatformAgent(client)

# Ejecución correcta
print('Ejecutando caso correcto...')
correct_answer = agent.execute('¿Cuál es la capital de Alemania?')
print(correct_answer)

# Ejecución con error simulado
print('Ejecutando caso con error simulado...')
client.log_event('model.call', {
    'session_id': 'error-12345',
    'model': 'mock-llm-v2',
    'response': '',
    'latency_s': 0.12,
    'tokens_in': 20,
    'tokens_out': 0,
    'cost_estimated': 0.0004,
    'error': 'timeout',
})
client.log_event('response.generated', {
    'session_id': 'error-12345',
    'final_answer': None,
    'status': 'failed',
})

# Mostrar las trazas como tabla
client.display()

## Interpretación de la salida

En esta tabla vemos cómo una plataforma especializada registra: 
- sesión y pregunta original,
- prompt usado,
- llamada al modelo con latencia, tokens y costo,
- resultado final y estado.

Esto permite comparar ejecuciones buenas y malas de forma estructurada, sin depender únicamente de logs dispersos.

: {
: {
: 
3
, 
: 
, 
: 
},
: {
: 
}
: 4,
: 5